# CRISP-DM Fase 2: Data Understanding - Exploración Informática y Estadística (EDA)
## Dataset: IDEAM Observaciones Telemétricas en Tiempo Real (API 57sv-p2fu)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 2: Exploratory Data Analysis) | **Fase PDCO**: DEVELOPMENT  
**Estándares Aplicados**: ISO/IEC 25010 (Data Quality Metrics), Control Estadístico de Procesos (SPC Nelson Rules).

---

### 1. Objetivos del Análisis Exploratorio de Datos (EDA)
El EDA sistemático persigue dos metas complementarias:
1. **Diagnóstico Informático**: Identificar tipos de datos, consumo de memoria, cardinalidad y patrones de datos faltantes bajo la taxonomía de Donald Rubin:
   * **MCAR** (*Missing Completely at Random*): La ausencia es independiente de cualquier variable observada o no observada.
   * **MAR** (*Missing at Random*): La probabilidad de ausencia depende de otras variables observables.
   * **MNAR** (*Missing Not at Random*): La ausencia depende del valor intrínseco de la propia variable no observada.
2. **Diagnóstico Estadístico**: Evaluar la forma funcional de la distribución empírica mediante los **cuatro momentos estadísticos de Pearson** y detectar valores atípicos mediante **Control Estadístico de Procesos (SPC)**.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN ROBUSTA DE RUTAS DEL PROYECTO (PEP 8)]
# Compatible con Local Jupyter, VSCode, Colab, Databricks Git Folders & Repos
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Búsqueda dinámica ascendente de la raíz del proyecto (donde reside src/ y metadata.json)
CURRENT_DIR = Path(".").resolve()
APP_ROOT = None

for candidate in [CURRENT_DIR, CURRENT_DIR.parent, CURRENT_DIR.parent.parent, CURRENT_DIR.parent.parent.parent]:
    if (candidate / "src" / "database").exists() or (candidate / "metadata.json").exists():
        APP_ROOT = candidate
        break

if APP_ROOT is None:
    candidate = CURRENT_DIR
    while candidate.parent != candidate:
        if (candidate / "src" / "database").exists():
            APP_ROOT = candidate
            break
        candidate = candidate.parent

if APP_ROOT is None:
    APP_ROOT = CURRENT_DIR

if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno e importaciones inicializadas exitosamente.")
print(f"📍 Raíz del proyecto detectada: {APP_ROOT}")


### 2. Perfilamiento Informático y Completitud
Examinamos la estructura tabular, el footprint en memoria RAM y el porcentaje de completitud por columna para planificar las reglas de limpieza y validación.


In [ ]:
# ==============================================================================
# [FASE 3: PERFILAMIENTO INFORMÁTICO, COMPLETITUD Y MATRIZ MISSINGNO]
# ==============================================================================
import matplotlib.pyplot as plt
import missingno as msno
from src.database.db_manager import DatabaseManager

parquet_file = APP_ROOT / "data/processed/ideam_telemetria_realtime.parquet"
if parquet_file.exists():
    df = pd.read_parquet(parquet_file)
else:
    db = DatabaseManager(str(APP_ROOT / "data" / "processed" / "agrostats_lakehouse.db"))
    df = db.query("SELECT * FROM ideam_telemetria_realtime")

print(f"📊 Dimensiones del Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"💾 Consumo en Memoria RAM: {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")

print("\n--- DIAGNÓSTICO DE COMPLETITUD Y NULOS POR VARIABLE ---")
missing_df = pd.DataFrame({
    "tipo_dato": df.dtypes,
    "nulos": df.isnull().sum(),
    "nulos_pct": (df.isnull().sum() / len(df)) * 100,
    "valores_unicos": df.nunique()
})
display(missing_df)

# Perfil de Ausencias con Missingno
if len(df) >= 2 and len(df.columns) > 1:
    fig, ax = plt.subplots(figsize=(14, 5))
    sample_df = df.head(500) if len(df) > 500 else df
    msno.matrix(sample_df, ax=ax, sparkline=False, color=(0.18, 0.38, 0.58), fontsize=10)
    ax.set_title(f"Perfil de Ausencias (Missingno Matrix): IDEAM Observaciones Telemétricas en Tiempo Real (API 57sv-p2fu)", fontsize=12, fontweight="bold", pad=15)
    plt.tight_layout()
    plt.show()


### 3. Estimación de Momentos Duales, Ajuste MLE de Distribuciones y Regla 1 de Nelson
Para la variable cuantitativa principal, calculamos:
1. **Momentos Paramétricos vs. Robustos**:
   * Paramétricos: Media ($\\mu$), Desviación Estándar ($\\sigma$), Asimetría de Fisher-Pearson ($S$), Curtosis ($K$).
   * No Paramétricos: Mediana ($Med$), Rango Intercuartílico ($IQR$), Desviación Absoluta de la Mediana ($MAD$), Media Recortada al 10%.
2. **Ajuste de Distribuciones Teóricas (MLE)**:
   * Ajustamos las distribuciones Normal, Lognormal, Gamma, Exponencial y Weibull.
   * Realizamos el test de bondad de ajuste de **Kolmogorov-Smirnov** y seleccionamos el mejor modelo mediante el Criterio de Información de Akaike (**AIC**).
3. **Control Estadístico de Procesos (SPC - Regla 1 de Nelson)**: Identifica observaciones a más de tres desviaciones estándar de la media ($|z| > 3$).


In [ ]:
# ==============================================================================
# [FASE 3: DIAGNÓSTICO ESTADÍSTICO DE DISTRIBUCIONES (MLE), MOMENTOS DUALES Y SPC]
# ==============================================================================
from scipy import stats
from src.modeling.sarimax_model import AgroModeler
from src.modeling.statistical_profiler import StatisticalProfiler

target_col = "valorobservado"
if target_col in df.columns and pd.api.types.is_numeric_dtype(df[target_col]):
    s = df[target_col].dropna()
    
    # 1. Momentos Estadísticos Duales
    moments = StatisticalProfiler.compute_dual_moments(s)
    print("--- 1. MOMENTOS ESTADÍSTICOS DUALES (PARAMÉTRICOS VS ROBUSTOS) ---")
    print(f"• Media Paramétrica (μ): {moments['media_mu']:,.4f} | Mediana Robusta (Med): {moments['mediana']:,.4f}")
    print(f"• Desviación Estándar (σ): {moments['desviacion_sigma']:,.4f} | MAD Robusta: {moments['mad']:,.4f}")
    print(f"• Rango Intercuartílico (IQR): {moments['iqr']:,.4f} | Media Recortada 10%: {moments['media_recortada_10']:,.4f}")
    print(f"• Coeficiente de Asimetría (Skewness): {moments['asimetria_skewness']:.4f}")
    print(f"• Curtosis: {moments['curtosis']:.4f}")
    print(f"• Coeficiente de Variación (CV): {moments['coef_variacion_cv']:.2%} | RSD Robusto (MAD/Med): {moments['rsd_mad_robusto']:.2%}")
    
    # 2. Ajuste de Distribuciones Teóricas MLE
    fits = StatisticalProfiler.fit_probability_distributions(s)
    print("\n--- 2. AJUSTE DE DISTRIBUCIONES TEÓRICAS (MAXIMUM LIKELIHOOD ESTIMATION) ---")
    print(f"• Distribución Óptima Seleccionada (Mínimo AIC): {fits['mejor_distribucion']}")
    for dist_name, res in fits.get('comparativa_completa', {}).items():
        print(f"  - {dist_name.capitalize()}: KS Stat={res['ks_stat']:.4f}, p-val={res['ks_pvalue']:.4e}, AIC={res['aic']:,.2f}")
    
    # 3. Control Estadístico de Procesos (SPC)
    print("\n--- 3. DETECCIÓN DE ANOMALÍAS (SPC REGLA 1 DE NELSON) ---")
    anomalies = AgroModeler.detect_nelson_anomalies(s)
    print(f"• Observaciones fuera de control (|z| > 3): {anomalies.sum()} de {len(s)} ({(anomalies.sum()/len(s)):.2%})")
else:
    print("ℹ️ La columna 'valorobservado' es textual o estructurada. Evaluando cardinalidad categórica:")
    print(df.describe(include='all'))


### 4. Visualización Diagnóstica de Distribución y Vallas de Tukey
Construimos un panel de alta resolución con tres representaciones simultáneas:
1. **Histograma Empírico con Densidad Kernel (KDE)** contrastando Media y Mediana.
2. **Q-Q Plot Normal** para inspección visual de curtosis y colas pesadas.
3. **Diagrama de Caja y Violín (Boxplot)** señalando cuartiles $Q_1, Q_2, Q_3$ y vallas de Tukey ($1.5 \\times IQR$).


In [ ]:
# ==============================================================================
# [FASE 3: PANEL VISUAL DE RIGOR ESTADÍSTICO Y DISTRIBUCIONES]
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

target_col = "valorobservado"
if target_col in df.columns and pd.api.types.is_numeric_dtype(df[target_col]):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    s = df[target_col].dropna()
    
    # Subplot 1: Histograma y KDE
    sns.histplot(s, kde=True, ax=axes[0], color="#2b5c8f", bins=30, stat="density")
    axes[0].axvline(s.mean(), color="red", linestyle="--", label=f"Media: {s.mean():,.2f}")
    axes[0].axvline(s.median(), color="green", linestyle="-", label=f"Mediana: {s.median():,.2f}")
    axes[0].set_title("Distribución Empírica y KDE", fontweight="bold")
    axes[0].legend()
    
    # Subplot 2: Q-Q Plot
    stats.probplot(s, dist="norm", plot=axes[1])
    axes[1].get_lines()[0].set_markerfacecolor('#e74c3c')
    axes[1].get_lines()[0].set_markeredgecolor('#c0392b')
    axes[1].get_lines()[1].set_color('#2c3e50')
    axes[1].set_title("Q-Q Plot vs. Normal Teórica", fontweight="bold")
    
    # Subplot 3: Boxplot y Violín
    sns.violinplot(y=s, ax=axes[2], color="#ecf0f1", inner=None)
    sns.boxplot(y=s, ax=axes[2], width=0.3, color="#3498db", fliersize=3)
    axes[2].set_title("Dispersión, Boxplot y Violín", fontweight="bold")
    
    plt.tight_layout()
    plt.show()


### 5. Diagnóstico Final del EDA y Recomendaciones para Data Preparation
1. **Tratamiento de Asimetrías**: Si el test Jarque-Bera rechaza la normalidad ($p < 0.05$), se hace imperativo el uso de estimadores no paramétricos (Mediana, Theil-Sen, Spearman).
2. **Siguiente Paso**: Proceder a `04_ingestion_como_dataframe.ipynb` para aplicar los Quality Gates automáticos de tipificación y completitud.
